# Module 5: Diagrams that do not lie

Your agent can read your account now, so the natural next step is to draw it. But asking a
small model to "diagram my cluster" with a generic drawing tool goes wrong: it has to read the
cluster, parse nested JSON, and hand-build the nodes and edges, and a 7B does not chain that
reliably. It draws a confident, wrong picture. In this module you will see the fabrication, then
build a deterministic tool that reads the account and renders the diagram in code, so the model
makes one reliable call.

## Learning objectives
- See a small model fabricate an architecture diagram when it has to read and build it itself
- Understand why read then parse then render is too long a chain for a 7B
- Build a deterministic diagram tool by hand: read the cluster, build the graph, render a PNG
- Wire it into the agent as `diagram_lke_cluster`, one reliable call with real data
- Add a second one, `diagram_network`, for the traffic flow
- Know when the generic diagram tool is still the right choice

## Prerequisites
- Finished Modules 1 through 4
- The same vLLM endpoint and a `LINODE_TOKEN` with at least one LKE cluster to draw
- The `graphviz` Python package and the `dot` binary

  ```bash
  pip install graphviz        # then install the system 'dot' binary:
  # macOS:  brew install graphviz
  # Debian: apt-get install graphviz
  ```
- About 25 minutes

References: [Strands Agents](https://strandsagents.com) · [graphviz](https://graphviz.org) · [Linode API: LKE](https://techdocs.akamai.com/linode-api/reference/get-lke-clusters) · [akamai-cloud-mcp](https://pypi.org/project/akamai-cloud-mcp/)

## Diagram design basics

A diagram from real data is three steps: read the resource, transform it into nodes and edges,
then render. The question is who does each step.

When you hand all three to a small model through a generic diagram tool, it has to read nested
JSON, hold it in context, and emit the exact structure the tool wants. A 7B drops or invents
parts of that chain. When you do the read and the build in code and leave the model one job,
"call this tool," the picture is always real.

![The agent makes one call to diagram_lke_cluster, which reads the cluster and renders a real PNG](../images/05_diagrams_architecture.png)

## 1. Setup

We use `httpx` to read the account directly and `graphviz` to render. No new agent packages
beyond the ones from the earlier modules. We reinstall here so this notebook stands on its own.
Run this notebook from the `workshop/05_diagrams` directory so the rendered PNGs land in a local
`diagrams/` folder.

In [ ]:
%pip install -q "strands-agents[openai]" strands-agents-tools httpx python-dotenv graphviz

In [ ]:
import os
import json

import httpx
import graphviz
from dotenv import load_dotenv
from strands import Agent, tool
from strands.models.openai import OpenAIModel

load_dotenv()

BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
MODEL_ID = os.getenv("VLLM_MODEL_ID", "Qwen/Qwen2.5-7B-Instruct")
API_KEY = os.getenv("VLLM_API_KEY", "placeholder")
LINODE_TOKEN = os.getenv("LINODE_TOKEN", "")

LINODE_API = "https://api.linode.com/v4"


def get_all(path, params=None, max_pages=10):
    """GET a paginated Linode collection and return all 'data' items."""
    items, page = [], 1
    params = dict(params or {})
    headers = {"Authorization": "Bearer " + LINODE_TOKEN}
    with httpx.Client(timeout=15.0) as client:
        while page <= max_pages:
            params["page"] = page
            r = client.get(LINODE_API + path, headers=headers, params=params)
            r.raise_for_status()
            body = r.json()
            items.extend(body.get("data", []))
            if page >= body.get("pages", 1):
                break
            page += 1
    return items


print(f"Model: {MODEL_ID}")
print(f"Endpoint: {BASE_URL}")
print("LKE clusters in this account:", len(get_all("/lke/clusters")))

## 2. Configure the model

The same raw HTTP helper from the earlier modules, so we can ask the model directly and see what
it produces before any tool gets involved.

In [ ]:
def chat(prompt, temperature=0.3):
    """One raw call to the model, returns the text."""
    resp = httpx.post(
        BASE_URL + "/chat/completions",
        headers={"Authorization": "Bearer " + API_KEY},
        json={"model": MODEL_ID, "messages": [{"role": "user", "content": prompt}], "temperature": temperature},
        timeout=120,
    )
    return resp.json()["choices"][0]["message"]["content"]


model = OpenAIModel(
    client_args={"api_key": API_KEY, "base_url": BASE_URL},
    model_id=MODEL_ID,
    params={"temperature": 0.2},
)
print("Model:", MODEL_ID)

## 3. The generic-tool path fumbles

Here is what a generic diagram tool asks of the model: read the account, figure out the cluster,
and emit the nodes and edges itself. We give it only the cluster labels, then ask it to build
"my GPU cluster" with the node pools and their GPU models and counts. It has no pool data, so it
fills the gap by inventing one. Watch it produce real-looking node pools that are not in your
account.

In [ ]:
clusters = get_all("/lke/clusters")
labels = [c.get("label") for c in clusters]
print("Real cluster labels:", labels, "\n")

prompt = (
    "Here are my LKE clusters: " + str(labels) + ".\n"
    "Draw 'my GPU cluster' as a diagram. Output JSON with a control plane node and one node "
    "for each GPU node pool, where every pool node has its GPU model and node count."
)
print(chat(prompt))

The pools it listed are invented. It never read them, because it could not in one step, so it
guessed. The diagram is confidently wrong, and a developer who trusts it ships the wrong picture.

## 4. Why: read then parse then build is too long a chain

The model is not bad at drawing. It is bad at orchestrating read, then parse, then build, all in
one pass. Prove it: hand the same model clean, pre-extracted JSON, the exact data the read step
would have produced, and ask only for the drawing. Now it gets it right.

In [ ]:
clean = {
    "cluster": "lke-gpu-demo",
    "region": "us-sea",
    "pools": [{"type": "g6-standard-2", "count": 2}, {"type": "g2-gpu-rtx4000a1-m", "count": 1}],
}
prompt = (
    "Here is a Kubernetes cluster as JSON:\n" + json.dumps(clean, indent=2) + "\n\n"
    "Output graphviz DOT with a control plane node and one node per pool labeled with its type "
    "and count. Only the DOT, nothing else."
)
print(chat(prompt))

Same model, correct picture. The failure was never the drawing. It was making the small model do
the read and the parse and the build together. So we move those two steps into code.

## 5. A deterministic tool by hand

This function does the whole job in code. It reads the clusters, picks a GPU cluster if there is
one, reads that cluster's node pools, then builds the graphviz graph node by node with the real
plan ids and counts, and renders a PNG. The model is not in the loop at all here.

In [ ]:
DIAGRAM_DIR = "diagrams"


def is_gpu(plan):
    return "gpu" in (plan or "").lower()


def render_lke_cluster(cluster_id=None):
    """Read an LKE cluster and its node pools and render them to a PNG. Returns path + summary."""
    clusters = get_all("/lke/clusters")
    if not clusters:
        return "No LKE clusters found in this account."

    # Pick the cluster: the named one, else a GPU cluster, else the first.
    cluster = None
    if cluster_id is not None:
        cluster = next((c for c in clusters if c.get("id") == cluster_id), None)
        if cluster is None:
            return "No LKE cluster with id " + str(cluster_id) + " in this account."
    else:
        for c in clusters:
            pools = get_all("/lke/clusters/" + str(c["id"]) + "/pools")
            if any(is_gpu(p.get("type", "")) for p in pools):
                cluster = c
                break
        cluster = cluster or clusters[0]

    cid = cluster["id"]
    label = cluster.get("label", str(cid))
    region = cluster.get("region", "?")
    k8s = cluster.get("k8s_version", "?")
    pools = get_all("/lke/clusters/" + str(cid) + "/pools")

    g = graphviz.Digraph(comment=label)
    g.attr(rankdir="TB")
    g.node("cp", "Control Plane\n" + label + "\nk8s " + str(k8s) + " | " + region,
           shape="box", style="filled", fillcolor="#cfe8ff")
    summary_pools = []
    for i, p in enumerate(pools):
        plan = p.get("type", "?")
        count = p.get("count", len(p.get("nodes", []) or []))
        fill = "#ffe0b3" if is_gpu(plan) else "#d7f0d0"
        g.node("pool%d" % i, "Node Pool\n" + plan + "\nx" + str(count) + " node(s)",
               shape="box", style="filled", fillcolor=fill)
        g.edge("cp", "pool%d" % i, label="manages")
        summary_pools.append(plan + " x" + str(count))

    os.makedirs(DIAGRAM_DIR, exist_ok=True)
    path = g.render(os.path.join(DIAGRAM_DIR, "lke-" + str(cid)), format="png", cleanup=True)
    pools_text = "; ".join(summary_pools) if summary_pools else "no node pools"
    summary = ("Cluster '" + label + "' (id " + str(cid) + ") in " + region + ", Kubernetes "
               + str(k8s) + ".\nControl plane manages " + str(len(pools)) + " node pool(s): " + pools_text + ".")
    return "Diagram saved to " + path + "\n" + summary


print(render_lke_cluster())

In [ ]:
from IPython.display import Image

# Show the rendered diagram. The path printed above ends in diagrams/lke-<id>.png.
clusters = get_all("/lke/clusters")
gpu = next((c for c in clusters if any(is_gpu(p.get("type", "")) for p in get_all("/lke/clusters/" + str(c["id"]) + "/pools"))), clusters[0])
Image("diagrams/lke-" + str(gpu["id"]) + ".png")

Every label is real: the cluster id, the Kubernetes version, the region, the plan ids, the node
counts. There is nothing for the model to invent, because the tool read it.

## 6. The same wired into the agent

Now wrap that exact function as a tool and give it to an agent. The tool's docstring tells the
model to make one call and not to read anything first. The system prompt is the diagramming rule:
to draw an LKE cluster, call `diagram_lke_cluster`, then show the path and summary it returns.

In [ ]:
@tool
def diagram_lke_cluster(cluster_id: int | None = None) -> str:
    """Draw an architecture diagram of an LKE (Kubernetes) cluster from real data.

    Use this whenever asked to draw or diagram an LKE / Kubernetes cluster. It reads the cluster
    and its node pools and renders the control plane and each node pool with its real plan id and
    node count. You do not need to read anything first; this tool does it for you in one call.

    Args:
        cluster_id: The LKE cluster id. If omitted, a GPU cluster is chosen when one exists.
    Returns:
        The path to the rendered PNG and a short text summary of the cluster.
    """
    return render_lke_cluster(cluster_id)


DIAGRAM_PROMPT = (
    "You are an Akamai Cloud Solutions Architect. To draw an LKE or Kubernetes cluster, call "
    "diagram_lke_cluster. It reads the cluster and renders it in one call, so do not read or "
    "build nodes yourself. Then show the returned image path and the summary. No em-dashes."
)

# callback_handler=None so the model's streaming text does not print over our result.
agent = Agent(model=model, system_prompt=DIAGRAM_PROMPT, tools=[diagram_lke_cluster], callback_handler=None)
result = agent("Diagram my GPU cluster.")
print(result)

One tool call, real data, the same PNG. The model went from fabricating the cluster to drawing
the true one, because the hard part now lives in code. Notice the model wrote no nodes, no edges,
no JSON. It made one call. The read, the parse, and the build all live in the tool.

## 7. A second deterministic tool: diagram_network

The same idea covers the network. `render_network` reads the NodeBalancers, the firewalls in
front of them, and the Linodes behind them, and draws the traffic flow. The default view is
firewall to NodeBalancer to backend Linodes. Pass `all_firewalls=True` to instead draw every
firewall and what it protects.

In [ ]:
def render_network(all_firewalls=False):
    """Read the network and render the traffic flow (or every firewall). Returns path + summary."""
    def safe(path):
        try:
            return get_all(path)
        except Exception:
            return []

    firewalls = safe("/networking/firewalls")
    fw_devices, nb_firewalls = {}, {}
    for fw in firewalls:
        devices = safe("/networking/firewalls/" + str(fw.get("id")) + "/devices")
        fw_devices[fw.get("id")] = devices
        for d in devices:
            ent = d.get("entity") or {}
            if ent.get("type") == "nodebalancer":
                nb_firewalls.setdefault(ent.get("id"), []).append(fw)

    os.makedirs(DIAGRAM_DIR, exist_ok=True)

    if all_firewalls:
        if not firewalls:
            return "No firewalls found in this account."
        g = graphviz.Digraph(comment="firewalls")
        g.attr(rankdir="TB")
        for fw in firewalls:
            fwid = "fw-" + str(fw.get("id"))
            g.node(fwid, "Firewall\n" + str(fw.get("label", "?")) + "\n" + str(fw.get("status", "")),
                   shape="box", style="filled", fillcolor="#ffcccc")
            for d in fw_devices.get(fw.get("id"), []):
                ent = d.get("entity") or {}
                etype, eid = ent.get("type"), ent.get("id")
                prefix = "nb" if etype == "nodebalancer" else "linode" if etype == "linode" else str(etype)
                target = prefix + "-" + str(eid)
                g.node(target, str(ent.get("label", eid)) + "\n(" + str(etype) + ")", shape="ellipse",
                       style="filled", fillcolor="#ffe0b3" if etype == "nodebalancer" else "#d7f0d0")
                g.edge(fwid, target, label="protects", style="dashed")
        path = g.render(os.path.join(DIAGRAM_DIR, "firewalls"), format="png", cleanup=True)
        return "Diagram saved to " + path + "\n" + str(len(firewalls)) + " firewall(s) and what each protects."

    nodebalancers = safe("/nodebalancers")
    instances = safe("/linode/instances")
    ip_to_inst = {}
    for inst in instances:
        for ip in inst.get("ipv4") or []:
            ip_to_inst[ip] = inst
    if not nodebalancers:
        return "No NodeBalancers found, so there is no traffic flow to draw."

    g = graphviz.Digraph(comment="network")
    g.attr(rankdir="LR")
    fw_drawn, fronted, total_backends = set(), 0, 0
    for nb in nodebalancers:
        nbid = nb.get("id")
        nbnode = "nb-" + str(nbid)
        g.node(nbnode, "NodeBalancer\n" + str(nb.get("label", "?")) + "\n" + str(nb.get("hostname", "")),
               shape="box", style="filled", fillcolor="#ffe0b3")
        for fw in nb_firewalls.get(nbid, []):
            fwnode = "fw-" + str(fw.get("id"))
            if fwnode not in fw_drawn:
                g.node(fwnode, "Firewall\n" + str(fw.get("label", "?")), shape="box", style="filled", fillcolor="#ffcccc")
                fw_drawn.add(fwnode)
            g.edge(fwnode, nbnode, label="allows")
        if nb_firewalls.get(nbid):
            fronted += 1
        seen = set()
        for cfg in safe("/nodebalancers/" + str(nbid) + "/configs"):
            for node in safe("/nodebalancers/" + str(nbid) + "/configs/" + str(cfg.get("id")) + "/nodes"):
                ip = (node.get("address") or "").split(":")[0]
                if not ip:
                    continue
                inst = ip_to_inst.get(ip)
                key = inst.get("id") if inst else ip
                if key in seen:
                    continue
                seen.add(key)
                total_backends += 1
                if inst:
                    bnode = "linode-" + str(inst.get("id"))
                    g.node(bnode, str(inst.get("label", "Linode")) + "\n(" + str(inst.get("id")) + ")",
                           shape="ellipse", style="filled", fillcolor="#d7f0d0")
                else:
                    bnode = "ip-" + ip
                    g.node(bnode, ip, shape="ellipse")
                g.edge(nbnode, bnode, label="backend")

    path = g.render(os.path.join(DIAGRAM_DIR, "network"), format="png", cleanup=True)
    return ("Diagram saved to " + path + "\n" + str(len(nodebalancers)) + " NodeBalancer(s), "
            + str(fronted) + " fronted by a firewall, " + str(total_backends) + " backend Linode(s).")


print(render_network())

In [ ]:
from IPython.display import Image
Image("diagrams/network.png")

Wrapping `render_network` as a `@tool` is identical to section 6: add the docstring, hand it to
the agent, and "diagram my network" becomes one reliable call. Same pattern, second picture.

## 8. Keep the generic diagram tool for designed diagrams

Deterministic tools win when there is something real to read. But sometimes there is nothing to
read: you are sketching a solution you have not built yet, like the multi-agent system in Module
7. There, the user describes the components, so the generic diagram tool is the right call. The
rule mirrors Module 7's "when not to wrap a tool in an agent": build a deterministic tool when
the data is real and the shape is known; use the generic tool when the model is drawing an idea.

## Things to know

- **The dot binary is separate.** The `graphviz` Python package only builds the graph. It shells
  out to the system `dot` binary to render, so without `dot` installed the render call raises.
- **Streaming before the summary.** Set `callback_handler=None` on the agent, or the model's
  streamed text prints over the path and summary you want to show.
- **Pagination.** `get_all` caps at `max_pages=10`. A very large account can truncate, so raise the
  cap if you have more than ten pages of clusters, NodeBalancers, or instances.
- **Read-only.** These tools only GET from the account. They never change anything, so they are
  safe to hand to the agent without the approval gate from Module 3.

## Try it yourself

1. **Draw every firewall.** Run `render_network(all_firewalls=True)` and display
   `diagrams/firewalls.png`. Compare it to the default traffic-flow view.
2. **Pick a cluster.** Pass a specific `cluster_id` to `diagram_lke_cluster` and draw a non-GPU
   cluster.
3. **Retry the fabrication.** Ask the agent the question from section 3 ("diagram my GPU cluster")
   with the real tool loaded, and confirm it calls the tool instead of inventing pools.

**Stretch:** wrap `render_network` as a `@tool`, give it to the agent alongside `diagram_lke_cluster`,
and ask "diagram my network."

In [ ]:
# Change all_firewalls (or pass a cluster_id to diagram_lke_cluster), then run the cell.
from IPython.display import Image

print(render_network(all_firewalls=True))
Image("diagrams/firewalls.png")

## Summary

- A diagram from real data is read, then transform, then render. A small model is unreliable when
  it has to do all three in one pass, so it fabricates.
- The same model draws correctly when you hand it clean, pre-extracted data. The failure is the
  orchestration, not the drawing.
- A deterministic tool does the read and the build in code and leaves the model one call. The
  picture is always real: `diagram_lke_cluster`, `diagram_network`.
- Keep the generic diagram tool for solutions you are designing, where there is nothing to read.

## Next

**Module 6: Evals.** The agent feels right, but "it felt fine" is not a measurement. Next you will
score its behavior, find a real gap, fix it, and watch the number rise.